# Portion 3 · Prepare embeddings (slow-connection path)

The five Portion 2 embedding files are 5.2 GB each (26 GB total). LightGBM only uses the
first 256 columns, so this notebook slices them on Drive and writes 13 GB of 2.6 GB files.
Download that folder to the PC instead of the originals.

Skip this notebook if your connection can handle 26 GB: download the originals and run
`python -m Portion3.prepare_embeddings slice --source-dir ... --out-dir ...` locally.

In [ ]:
import subprocess, sys
from pathlib import Path

REPO = Path("/content/TabBERT_AML")
if not (REPO / "Portion3").exists():
    subprocess.run(["git", "clone", "--branch", "portion3",
                    "https://github.com/ted-cc6/TabBERT_AML.git", str(REPO)], check=True)
    subprocess.run(["git", "lfs", "install"], cwd=REPO, check=True)
    subprocess.run(["git", "lfs", "pull"], cwd=REPO, check=True)

from google.colab import drive
drive.mount("/content/drive")
SOURCE = Path("/content/drive/MyDrive/TabBERT_AML/portion2_runs")
OUT = Path("/content/drive/MyDrive/TabBERT_AML/portion3_embeddings")
print("source:", SOURCE)
print("out:", OUT)

## Slice and verify

This reads each 5.2 GB file once and writes a 2.6 GB float16 copy. It checks row counts,
duplicate ids, coverage against `window_index`, and finiteness first.

In [ ]:
runs = ["main_seed42", "main_seed43", "main_seed44", "abl_whole_mask_seed42", "abl_no_dt_seed42"]
cmd = [sys.executable, "-m", "Portion3.prepare_embeddings", "slice",
       "--source-dir", str(SOURCE), "--out-dir", str(OUT), "--runs", *runs]
print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO, check=True)

## Download

Download the `portion3_embeddings` folder (about 13 GB) and put it on the PC, then run:

```
python -m Portion3.prepare_embeddings check --source-dir <folder>
python -m Portion3.portion3_baselines lightgbm-embeddings --embeddings-dir <folder>
```